# T002 · 感知机（Perceptron）

Colab 跑这份 notebook 即可。详细解释见同目录 `T002-感知机-学习笔记.md`。

## Step A · 生成 toy 数据

两类高斯簇（均值拉开）保证线性可分；标签直接用 {-1,+1}。
注意：`sklearn.make_classification(class_sep=1.0)` 不保证可分。

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
X_pos = rng.normal(loc=[2.0, 2.0], scale=0.55, size=(100, 2))
X_neg = rng.normal(loc=[-2.0, -2.0], scale=0.55, size=(100, 2))
X = np.vstack([X_pos, X_neg])
y = np.array([1] * 100 + [-1] * 100, dtype=float)
idx = rng.permutation(200)
X, y = X[idx], y[idx]

print("X shape:", X.shape)            # (200, 2)
print("y unique:", set(y))            # {-1.0, 1.0}
print("每类样本数:", {c: int((y == c).sum()) for c in set(y)})

## Step B · 感知机类（PLA 算法）

`fit` 每轮遍历所有样本，逐个判断，错就把权重往正确方向推一步；全对就提前停。

`predict` 输出 +1 或 -1。

In [ ]:
class Perceptron:
    """Rosenblatt 1958 感知机，标准 PLA。标签约定 {-1,+1}。"""
    def __init__(self, lr=1.0, max_epochs=100):
        self.lr = lr
        self.max_epochs = max_epochs

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.errors_history = []

        for epoch in range(self.max_epochs):
            errors = 0
            for xi, yi in zip(X, y):
                pred = np.sign(np.dot(xi, self.w) + self.b)
                if pred == 0:
                    pred = -1.0
                if pred != yi:
                    self.w += self.lr * yi * xi
                    self.b += self.lr * yi
                    errors += 1
            self.errors_history.append(errors)
            if errors == 0:
                break
        return self

    def predict(self, X):
        preds = np.sign(X @ self.w + self.b)
        preds[preds == 0] = -1.0
        return preds

## Step C · 训练 + 评估

7:3 划分训练/测试，stratify 保证分布一致。预期测试准确率 ≥ 95%（数据线性可分）。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = Perceptron(lr=1.0, max_epochs=100)
model.fit(X_train, y_train)

train_acc = (model.predict(X_train) == y_train).mean()
test_acc  = (model.predict(X_test)  == y_test).mean()

print(f"训练准确率：{train_acc:.2%}")
print(f"测试准确率：{test_acc:.2%}")
print(f"收敛轮数：  {len(model.errors_history)}")
print(f"最终权重：  w = {model.w}")
print(f"最终偏置：  b = {model.b:.4f}")

## Step D · 画决策边界

两类点用不同颜色画散点，再画直线 $w \cdot x + b = 0$，肉眼能直接看出模型切开了两堆。

In [ ]:
import matplotlib.pyplot as plt
import os
os.makedirs("figures", exist_ok=True)

plt.figure(figsize=(8, 6))
plt.scatter(X[y == 1, 0],  X[y == 1, 1],  c="royalblue", label="+1", alpha=0.7)
plt.scatter(X[y == -1, 0], X[y == -1, 1], c="tomato",    label="-1", alpha=0.7)

# 决策直线 w·x + b = 0  →  x2 = -(w1·x1 + b) / w2
x1_line = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
x2_line = -(model.w[0] * x1_line + model.b) / model.w[1]
plt.plot(x1_line, x2_line, "k--", lw=2, label="决策边界")

plt.xlabel("特征 1")
plt.ylabel("特征 2")
plt.title(f"感知机决策边界  ·  测试准确率 {test_acc:.2%}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/decision_boundary.png", dpi=120)
plt.show()

## Step E · 画收敛曲线

横轴 epoch，纵轴每轮错分样本数。线性可分时曲线应一路降到 0。

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(model.errors_history) + 1), model.errors_history, "o-")
plt.xlabel("Epoch")
plt.ylabel("错分样本数")
plt.title("感知机训练收敛曲线")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/convergence.png", dpi=120)
plt.show()

## ✅ 做完这个实验你应该记住三件事

| 序号 | 知识点 | 一句话总结 |
|------|--------|-----------|
| 1 | **Novikoff 定理** | 只要线性可分，PLA **一定收敛**，错分次数有上界 |
| 2 | **感知机 vs 逻辑回归** | 前者硬分类（±1），后者软输出（概率），是近亲 |
| 3 | **深度学习的起源** | 加隐藏层 + 激活函数 → MLP → 就是神经网络了 |

💡 下一步：把 `sign` 换成 `sigmoid`，你就写出了逻辑回归！